In [21]:
import pandas as pd

import numpy as np
import pandas as pd
from collections import deque


EPS = 1e-8


def _safe_array(x):
    x = np.asarray(x, dtype=np.float64)
    x = x[np.isfinite(x)]
    return x


def _safe_std(x):
    x = _safe_array(x)
    if len(x) <= 1:
        return 0.0
    return float(np.std(x, ddof=1))


def _safe_skew(x):
    x = _safe_array(x)
    if len(x) <= 2:
        return 0.0

    mu = np.mean(x)
    sd = np.std(x) + EPS
    z = (x - mu) / sd
    return float(np.mean(z ** 3))


def _safe_kurtosis(x):
    x = _safe_array(x)
    if len(x) <= 3:
        return 0.0

    mu = np.mean(x)
    sd = np.std(x) + EPS
    z = (x - mu) / sd
    return float(np.mean(z ** 4) - 3.0)


def _autocorr(x, lag=1):
    x = _safe_array(x)
    if len(x) <= lag + 1:
        return 0.0

    x1 = x[:-lag]
    x2 = x[lag:]

    s1 = np.std(x1)
    s2 = np.std(x2)

    if s1 < EPS or s2 < EPS:
        return 0.0

    return float(np.corrcoef(x1, x2)[0, 1])


def _slope_features(x):
    x = _safe_array(x)
    n = len(x)

    if n <= 2:
        return 0.0, 0.0

    t = np.arange(n, dtype=np.float64)
    t = (t - t.mean()) / (t.std() + EPS)

    y = (x - x.mean()) / (x.std() + EPS)

    slope = float(np.polyfit(t, y, deg=1)[0])

    y_hat = slope * t
    ss_res = np.sum((y - y_hat) ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2) + EPS
    r2 = float(1.0 - ss_res / ss_tot)

    return slope, r2


def _sign_change_rate(x):
    x = _safe_array(x)
    if len(x) <= 2:
        return 0.0

    dx = np.diff(x)
    signs = np.sign(dx)
    signs = signs[signs != 0]

    if len(signs) <= 1:
        return 0.0

    return float(np.mean(signs[1:] != signs[:-1]))


def _turning_point_rate(x):
    x = _safe_array(x)
    if len(x) <= 2:
        return 0.0

    dx1 = np.diff(x[:-1])
    dx2 = np.diff(x[1:])

    turning = (dx1 * dx2) < 0
    return float(np.mean(turning))


def _zero_crossing_rate(x):
    x = _safe_array(x)
    if len(x) <= 1:
        return 0.0

    x = x - np.mean(x)
    signs = np.sign(x)
    signs = signs[signs != 0]

    if len(signs) <= 1:
        return 0.0

    return float(np.mean(signs[1:] != signs[:-1]))


class StreamingBreakFeatureExtractor:
    """
    Causal feature extractor for the Crunch structural break task.

    At each online point:
        1. update internal rolling windows
        2. compute features using only historical data + online points seen so far
        3. return one feature dictionary

    This can be used in both training and inference.
    """

    def __init__(
        self,
        x_historical,
        windows=(8, 16, 32, 64, 128),
        cusum_drift=0.25,
        ewma_alpha=0.05,
    ):
        self.x_hist = _safe_array(x_historical)
        self.windows = list(windows)

        self.hist_mean = float(np.mean(self.x_hist))
        self.hist_std = float(np.std(self.x_hist, ddof=1)) if len(self.x_hist) > 1 else 0.0
        self.hist_std = max(self.hist_std, EPS)

        self.hist_median = float(np.median(self.x_hist))
        self.hist_iqr = float(np.quantile(self.x_hist, 0.75) - np.quantile(self.x_hist, 0.25))
        self.hist_iqr = max(self.hist_iqr, EPS)

        self.hist_mad = float(np.median(np.abs(self.x_hist - self.hist_median)))
        self.hist_mad = max(self.hist_mad, EPS)

        self.hist_quantiles = {
            q: float(np.quantile(self.x_hist, q))
            for q in [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
        }

        self.buffers = {
            W: deque(self.x_hist[-W:], maxlen=W)
            for W in self.windows
        }

        self.t = -1
        self.n_online_seen = 0

        self.cusum_pos = 0.0
        self.cusum_neg = 0.0
        self.cusum_drift = cusum_drift

        self.ewma_z = 0.0
        self.ewma_abs_z = 0.0
        self.ewma_alpha = ewma_alpha

        self.running_max_abs_z = 0.0
        self.running_mean_abs_z = 0.0

        self.consecutive_abs_z_gt_1 = 0
        self.consecutive_abs_z_gt_2 = 0
        self.consecutive_abs_z_gt_3 = 0

    def update(self, x_t):
        """
        Add one online point and return feature dict for the current time step.
        """

        x_t = float(x_t)
        self.t += 1
        self.n_online_seen += 1

        for W in self.windows:
            self.buffers[W].append(x_t)

        z_t = (x_t - self.hist_mean) / self.hist_std
        abs_z_t = abs(z_t)

        # Cumulative evidence features
        self.cusum_pos = max(0.0, self.cusum_pos + z_t - self.cusum_drift)
        self.cusum_neg = max(0.0, self.cusum_neg - z_t - self.cusum_drift)

        self.ewma_z = (1.0 - self.ewma_alpha) * self.ewma_z + self.ewma_alpha * z_t
        self.ewma_abs_z = (1.0 - self.ewma_alpha) * self.ewma_abs_z + self.ewma_alpha * abs_z_t

        self.running_max_abs_z = max(self.running_max_abs_z, abs_z_t)
        self.running_mean_abs_z += (abs_z_t - self.running_mean_abs_z) / self.n_online_seen

        if abs_z_t > 1:
            self.consecutive_abs_z_gt_1 += 1
        else:
            self.consecutive_abs_z_gt_1 = 0

        if abs_z_t > 2:
            self.consecutive_abs_z_gt_2 += 1
        else:
            self.consecutive_abs_z_gt_2 = 0

        if abs_z_t > 3:
            self.consecutive_abs_z_gt_3 += 1
        else:
            self.consecutive_abs_z_gt_3 = 0

        feats = {
            "t": self.t,
            "n_online_seen": self.n_online_seen,

            "last_value": x_t,
            "last_z": z_t,
            "last_abs_z": abs_z_t,

            "cusum_pos": self.cusum_pos,
            "cusum_neg": self.cusum_neg,
            "cusum_max": max(self.cusum_pos, self.cusum_neg),
            "cusum_diff": self.cusum_pos - self.cusum_neg,

            "ewma_z": self.ewma_z,
            "ewma_abs_z": self.ewma_abs_z,
            "running_max_abs_z": self.running_max_abs_z,
            "running_mean_abs_z": self.running_mean_abs_z,

            "consecutive_abs_z_gt_1": self.consecutive_abs_z_gt_1,
            "consecutive_abs_z_gt_2": self.consecutive_abs_z_gt_2,
            "consecutive_abs_z_gt_3": self.consecutive_abs_z_gt_3,
        }

        for W in self.windows:
            w = np.asarray(self.buffers[W], dtype=np.float64)

            feats.update(self._window_features(w, W))

        # Short-vs-long agreement features
        if len(self.windows) >= 2:
            Ws = sorted(self.windows)
            W_short = Ws[0]
            W_long = Ws[-1]

            feats[f"mean_diff_w{W_short}_w{W_long}"] = (
                feats[f"mean_w{W_short}"] - feats[f"mean_w{W_long}"]
            )
            feats[f"std_ratio_w{W_short}_w{W_long}"] = (
                feats[f"std_w{W_short}"] / (feats[f"std_w{W_long}"] + EPS)
            )
            feats[f"abs_z_mean_diff_w{W_short}_w{W_long}"] = (
                feats[f"abs_z_mean_w{W_short}"] - feats[f"abs_z_mean_w{W_long}"]
            )

        return feats

    def _window_features(self, w, W):
        mean_w = float(np.mean(w))
        std_w = _safe_std(w)
        median_w = float(np.median(w))

        q01 = float(np.quantile(w, 0.01))
        q05 = float(np.quantile(w, 0.05))
        q10 = float(np.quantile(w, 0.10))
        q25 = float(np.quantile(w, 0.25))
        q50 = float(np.quantile(w, 0.50))
        q75 = float(np.quantile(w, 0.75))
        q90 = float(np.quantile(w, 0.90))
        q95 = float(np.quantile(w, 0.95))
        q99 = float(np.quantile(w, 0.99))

        iqr_w = q75 - q25
        mad_w = float(np.median(np.abs(w - median_w)))

        z_w = (w - self.hist_mean) / self.hist_std
        abs_z_w = np.abs(z_w)

        slope, slope_r2 = _slope_features(w)

        diffs = np.diff(w)
        if len(diffs) == 0:
            mean_abs_diff = 0.0
            std_abs_diff = 0.0
            max_abs_diff = 0.0
            mean_sq_diff = 0.0
        else:
            abs_diffs = np.abs(diffs)
            mean_abs_diff = float(np.mean(abs_diffs))
            std_abs_diff = float(np.std(abs_diffs))
            max_abs_diff = float(np.max(abs_diffs))
            mean_sq_diff = float(np.mean(diffs ** 2))

        quantile_diffs = np.array([
            q01 - self.hist_quantiles[0.01],
            q05 - self.hist_quantiles[0.05],
            q10 - self.hist_quantiles[0.10],
            q25 - self.hist_quantiles[0.25],
            q50 - self.hist_quantiles[0.50],
            q75 - self.hist_quantiles[0.75],
            q90 - self.hist_quantiles[0.90],
            q95 - self.hist_quantiles[0.95],
            q99 - self.hist_quantiles[0.99],
        ])

        return {
            f"mean_w{W}": mean_w,
            f"std_w{W}": std_w,
            f"median_w{W}": median_w,
            f"min_w{W}": float(np.min(w)),
            f"max_w{W}": float(np.max(w)),
            f"range_w{W}": float(np.max(w) - np.min(w)),
            f"iqr_w{W}": float(iqr_w),
            f"mad_w{W}": float(mad_w),

            f"z_mean_w{W}": (mean_w - self.hist_mean) / self.hist_std,
            f"abs_z_mean_w{W}": abs((mean_w - self.hist_mean) / self.hist_std),
            f"z_median_w{W}": (median_w - self.hist_median) / self.hist_std,
            f"abs_z_median_w{W}": abs((median_w - self.hist_median) / self.hist_std),

            f"log_std_ratio_w{W}": float(np.log((std_w + EPS) / self.hist_std)),
            f"log_iqr_ratio_w{W}": float(np.log((iqr_w + EPS) / self.hist_iqr)),
            f"log_mad_ratio_w{W}": float(np.log((mad_w + EPS) / self.hist_mad)),

            f"max_abs_z_w{W}": float(np.max(abs_z_w)),
            f"mean_abs_z_w{W}": float(np.mean(abs_z_w)),
            f"q95_abs_z_w{W}": float(np.quantile(abs_z_w, 0.95)),
            f"frac_abs_z_gt_1_w{W}": float(np.mean(abs_z_w > 1.0)),
            f"frac_abs_z_gt_2_w{W}": float(np.mean(abs_z_w > 2.0)),
            f"frac_abs_z_gt_3_w{W}": float(np.mean(abs_z_w > 3.0)),

            f"q01_diff_w{W}": q01 - self.hist_quantiles[0.01],
            f"q05_diff_w{W}": q05 - self.hist_quantiles[0.05],
            f"q10_diff_w{W}": q10 - self.hist_quantiles[0.10],
            f"q25_diff_w{W}": q25 - self.hist_quantiles[0.25],
            f"q50_diff_w{W}": q50 - self.hist_quantiles[0.50],
            f"q75_diff_w{W}": q75 - self.hist_quantiles[0.75],
            f"q90_diff_w{W}": q90 - self.hist_quantiles[0.90],
            f"q95_diff_w{W}": q95 - self.hist_quantiles[0.95],
            f"q99_diff_w{W}": q99 - self.hist_quantiles[0.99],
            f"mean_abs_quantile_diff_w{W}": float(np.mean(np.abs(quantile_diffs))),
            f"max_abs_quantile_diff_w{W}": float(np.max(np.abs(quantile_diffs))),

            f"slope_w{W}": slope,
            f"abs_slope_w{W}": abs(slope),
            f"slope_r2_w{W}": slope_r2,

            f"mean_abs_diff_w{W}": mean_abs_diff,
            f"std_abs_diff_w{W}": std_abs_diff,
            f"max_abs_diff_w{W}": max_abs_diff,
            f"mean_sq_diff_w{W}": mean_sq_diff,

            f"autocorr_lag1_w{W}": _autocorr(w, lag=1),
            f"autocorr_lag2_w{W}": _autocorr(w, lag=2),
            f"autocorr_lag5_w{W}": _autocorr(w, lag=5),

            f"sign_change_rate_w{W}": _sign_change_rate(w),
            f"turning_point_rate_w{W}": _turning_point_rate(w),
            f"zero_crossing_rate_w{W}": _zero_crossing_rate(w),

            f"skew_w{W}": _safe_skew(w),
            f"kurtosis_w{W}": _safe_kurtosis(w),
        }


def extract_features_for_one_series(
    series_id,
    x_historical,
    x_online,
    tau=None,
    windows=(8, 16, 32, 64, 128),
):
    """
    Create one feature row per online point for one time series.

    If tau is given:
        target = 1 if t >= tau else 0

    If tau is None:
        target = 0 for all t
    """

    extractor = StreamingBreakFeatureExtractor(
        x_historical=x_historical,
        windows=windows,
    )

    rows = []

    for t, x_t in enumerate(x_online):
        row = extractor.update(x_t)

        row["series_id"] = series_id

        if tau is not None:
            row["target"] = int(t >= tau)
        else:
            row["target"] = 0

        rows.append(row)

    return pd.DataFrame(rows)


def extract_features_for_dataset(
    data,
    windows=(8, 16, 32, 64, 128),
    has_labels=True,
):
    """
    Build feature dataframe for many series.

    Expected training format:
        data = iterable of (series_id, x_historical, x_online, tau)

    Expected test/inference format:
        data = iterable of (series_id, x_historical, x_online)

    Returns:
        DataFrame with one row per online point.
    """

    dfs = []

    for item in data:
        if has_labels:
            series_id, x_historical, x_online, tau = item
        else:
            x_historical, x_online = item
            tau,series_id = None, None

        df_i = extract_features_for_one_series(
            series_id=series_id,
            x_historical=x_historical,
            x_online=x_online,
            tau=tau,
            windows=windows,
        )

        dfs.append(df_i)

    if len(dfs) == 0:
        return pd.DataFrame()

    return pd.concat(dfs, axis=0, ignore_index=True)

In [44]:
df_train = pd.read_parquet('dataset\X_train.parquet')
y_train = pd.read_parquet('dataset\Y_train.parquet')
y_train_index = pd.read_parquet('dataset\Y_train_index.parquet')

In [45]:
df_train

value  period
id   time                  
0    0    -0.783992       1
     1    -0.414595       1
     2     1.674117       1
     3     1.575443       1
     4    -0.520342       1
...             ...     ...
9999 4484 -2.220005       2
     4485 -1.863676       2
     4486 -1.043584       2
     4487 -1.746708       2
     4488 -0.563600       2

[35036464 rows x 2 columns]

In [47]:
y_train_index

,tau_index,tau
id,,
0,72,1264
1,-1,-1
2,-1,-1
3,-1,-1
4,-1,-1
...,...,...
9995,-1,-1
9996,-1,-1
9997,-1,-1


In [2]:
%pip install crunch-cli --upgrade --quiet --progress-bar off
!crunch setup-notebook structural-break-real-time VksSxD8K3Rs4NVtw9q2Z9sc1

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\Diego\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


crunch-cli, version 11.4.1
delete c:\Users\Diego\Stream-Anomaly-Detection\.crunchdao
you appear to have never submitted code before
data\X_train.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/X_train.parquet (218514418 bytes)
data\X_test.reduced.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/X_test.reduced.parquet (2587435 bytes)
data\y_train.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/y_train.parquet (8356193 bytes)
data\y_test.reduced.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/y_test.reduced.parquet (106299 bytes)
data\y_test_index.reduced.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/y_test_index.reduced.parquet (3672 bytes)
data\y_train_index.parquet: download from


  0%|          | 0.00/219M [00:00<?, ?iB/s]
  0%|          | 81.9k/219M [00:00<09:14, 394kiB/s]
  0%|          | 180k/219M [00:00<06:10, 589kiB/s] 
  0%|          | 246k/219M [00:00<06:35, 552kiB/s]
  0%|          | 360k/219M [00:00<04:59, 728kiB/s]
  0%|          | 508k/219M [00:00<03:55, 926kiB/s]
  0%|          | 721k/219M [00:00<02:55, 1.24MiB/s]
  0%|          | 999k/219M [00:00<02:10, 1.67MiB/s]
  1%|          | 1.43M/219M [00:00<01:30, 2.40MiB/s]
  1%|          | 1.97M/219M [00:01<01:06, 3.24MiB/s]
  1%|▏         | 2.74M/219M [00:01<00:48, 4.48MiB/s]
  2%|▏         | 3.85M/219M [00:01<00:33, 6.36MiB/s]
  2%|▏         | 5.37M/219M [00:01<00:24, 8.84MiB/s]
  3%|▎         | 7.54M/219M [00:01<00:16, 12.4MiB/s]
  5%|▍         | 10.5M/219M [00:01<00:12, 17.1MiB/s]
  6%|▌         | 12.5M/219M [00:01<00:11, 18.0MiB/s]
  7%|▋         | 15.4M/219M [00:01<00:09, 20.8MiB/s]
  8%|▊         | 17.5M/219M [00:01<00:09, 20.8MiB/s]
  9%|▉         | 20.3M/219M [00:02<00:08, 22.6MiB/s]
 10%|█     

In [ ]:
import crunch

# Load the Crunch Toolings (data loader, local tester, submitter).
crunch_tools = crunch.load_notebook()

# Load the data.
train_data, test_data = crunch_tools.load_data()

loaded inline runner with module: <module '__main__'>

cli version: 11.4.1
available ram: 39.36 gb
available cpu: 16 core
----


In [43]:
type(train_data[0])

tuple

In [31]:
len(train_data[0][1]), len(train_data[0][2]) #series_id, x_historical, x_online, tau = item

(1192, 416)

In [ ]:
total_samples = 0

for i in range(len(train_data)):
    
    total_samples += len(train_data[i][2])

In [35]:
total_samples

5036517

In [32]:
df = extract_features_for_dataset([train_data[0]])

In [33]:
df

,t,n_online_seen,last_value,last_z,last_abs_z,cusum_pos,cusum_neg,cusum_max,cusum_diff,ewma_z,...,sign_change_rate_w128,turning_point_rate_w128,zero_crossing_rate_w128,skew_w128,kurtosis_w128,mean_diff_w8_w128,std_ratio_w8_w128,abs_z_mean_diff_w8_w128,series_id,target
0,0,1,0.078301,0.078268,0.078268,0.000000,0.000000,0.000000,0.000000,0.003913,...,0.634921,0.634921,0.511811,-0.247000,-0.057773,0.061071,1.287024,0.061045,0,0
1,1,2,-0.917343,-0.916959,0.916959,0.000000,0.666959,0.666959,-0.666959,-0.042130,...,0.626984,0.626984,0.519685,-0.242406,-0.072273,0.012391,1.307442,0.012386,0,0
2,2,3,-2.740690,-2.739540,2.739540,0.000000,3.156498,3.156498,-3.156498,-0.177001,...,0.619048,0.619048,0.511811,-0.271711,-0.048390,-0.384110,1.494976,0.383949,0,0
3,3,4,-2.393864,-2.392860,2.392860,0.000000,5.299358,5.299358,-5.299358,-0.287794,...,0.626984,0.626984,0.503937,-0.256725,-0.098325,-0.725410,1.539954,0.725105,0,0
4,4,5,1.378991,1.378413,1.378413,1.128413,3.670945,3.670945,-2.542532,-0.204483,...,0.619048,0.619048,0.511811,-0.279388,-0.118573,-0.359464,1.605642,0.359313,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
411,411,412,-0.607715,-0.607460,0.607460,0.000000,0.357460,0.357460,-0.357460,-0.063060,...,0.650794,0.650794,0.511811,-0.448467,1.047422,-0.035702,0.331873,0.035687,0,1
412,412,413,0.561513,0.561277,0.561277,0.311277,0.000000,0.311277,0.311277,-0.031843,...,0.658730,0.658730,0.511811,-0.463607,1.121606,0.056057,0.393764,-0.056034,0,1
413,413,414,0.142339,0.142279,0.142279,0.203556,0.000000,0.203556,0.203556,-0.023137,...,0.666667,0.666667,0.511811,-0.474448,1.128340,0.137807,0.373214,-0.137750,0,1
414,414,415,1.506964,1.506332,1.506332,1.459888,0.000000,1.459888,1.459888,0.053336,...,0.666667,0.666667,0.543307,-0.491412,1.088142,0.337560,0.547353,0.045756,0,1


In [9]:
test_data[0]

(array([ 0.7460508 ,  0.3180033 , -0.2617024 , ..., -0.16977763,
        -0.40242085,  0.98709315], dtype=float32),
 array([ 0.16337058, -0.21031705, -0.80726594, -0.6090948 , -0.7839387 ,
        -0.39233282, -0.2142592 , -1.1675206 ,  0.6448548 ,  1.6431352 ,
         1.2142026 ,  1.8587648 ,  0.65687466,  1.2074723 ,  1.5644082 ,
         1.3740435 ,  0.60828876, -0.21032086, -0.53296304,  1.3771534 ,
         1.2185194 ,  0.06368195,  0.13970971,  2.4363353 ,  2.2333708 ,
         2.0944803 , -0.55147487, -0.36974475,  0.87466794,  0.02574355,
         0.79662085,  1.6546004 ,  1.1468886 ,  0.48607388, -0.46615317,
        -1.0801065 , -0.8939702 , -0.8296916 , -0.63111955, -0.8147199 ,
         0.38409317, -0.08493335,  0.04272092, -1.7975286 , -2.1518598 ,
        -2.1896098 , -1.3845453 , -1.310458  , -1.2056837 , -0.82971245,
        -0.87910944, -0.88630676, -0.97356045,  0.20638156,  0.7503751 ,
         0.73952174, -0.5531782 ,  0.8479921 ,  1.1756194 , -0.4394068 ,
        

In [19]:
df[df.series_id == 0]

,t,n_online_seen,last_value,last_z,last_abs_z,cusum_pos,cusum_neg,cusum_max,cusum_diff,ewma_z,...,sign_change_rate_w128,turning_point_rate_w128,zero_crossing_rate_w128,skew_w128,kurtosis_w128,mean_diff_w8_w128,std_ratio_w8_w128,abs_z_mean_diff_w8_w128,series_id,target
0,0,1,0.078301,0.078268,0.078268,0.000000,0.000000,0.000000,0.000000,0.003913,...,0.634921,0.634921,0.511811,-0.247000,-0.057773,0.061071,1.287024,0.061045,0,0
1,1,2,-0.917343,-0.916959,0.916959,0.000000,0.666959,0.666959,-0.666959,-0.042130,...,0.626984,0.626984,0.519685,-0.242406,-0.072273,0.012391,1.307442,0.012386,0,0
2,2,3,-2.740690,-2.739540,2.739540,0.000000,3.156498,3.156498,-3.156498,-0.177001,...,0.619048,0.619048,0.511811,-0.271711,-0.048390,-0.384110,1.494976,0.383949,0,0
3,3,4,-2.393864,-2.392860,2.392860,0.000000,5.299358,5.299358,-5.299358,-0.287794,...,0.626984,0.626984,0.503937,-0.256725,-0.098325,-0.725410,1.539954,0.725105,0,0
4,4,5,1.378991,1.378413,1.378413,1.128413,3.670945,3.670945,-2.542532,-0.204483,...,0.619048,0.619048,0.511811,-0.279388,-0.118573,-0.359464,1.605642,0.359313,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
411,411,412,-0.607715,-0.607460,0.607460,0.000000,0.357460,0.357460,-0.357460,-0.063060,...,0.650794,0.650794,0.511811,-0.448467,1.047422,-0.035702,0.331873,0.035687,0,1
412,412,413,0.561513,0.561277,0.561277,0.311277,0.000000,0.311277,0.311277,-0.031843,...,0.658730,0.658730,0.511811,-0.463607,1.121606,0.056057,0.393764,-0.056034,0,1
413,413,414,0.142339,0.142279,0.142279,0.203556,0.000000,0.203556,0.203556,-0.023137,...,0.666667,0.666667,0.511811,-0.474448,1.128340,0.137807,0.373214,-0.137750,0,1
414,414,415,1.506964,1.506332,1.506332,1.459888,0.000000,1.459888,1.459888,0.053336,...,0.666667,0.666667,0.543307,-0.491412,1.088142,0.337560,0.547353,0.045756,0,1
